<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #0284c7; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Expresiones, Contextos y Transformaciones 🎯
      </h1>
      <p style="margin: 6px 0 0 0; color: #0284c7; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Polars de Alto Rendimiento
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #0284c7; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 11 Extra
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #0284c7; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/11%20-%20Polars/01_Expresiones_Contextos_y_Transformaciones.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 1. El Paradigma de Expresiones de Polars (*Expressions API*) 🧠

El mayor superpoder de Polars es su **Lenguaje de Expresiones**. En lugar de mutar columnas paso a paso como en Pandas (`df['x'] = df['y'] + 1`), en Polars creamos **expresiones declarativas** (`pl.col('y') + 1`) que se envían a un optimizador multihilo.

### Los 3 Contextos Fundamentales:
1. **`select(...)`:** Proyecta y computa columnas independientes de forma paralela.
2. **`with_columns(...)`:** Añade o modifica columnas conservando todas las existentes.
3. **`filter(...)`:** Filtra filas vectorialmente evaluando predicados booleanos.

> 💡 Una **expresión** (`pl.col("x") * 2`) no ejecuta nada por sí sola: es una receta. Solo se evalúa cuando la colocas dentro de uno de los 3 contextos anteriores, y ahí Polars decide cómo repartirla entre los núcleos del procesador.

In [47]:
import polars as pl
import numpy as np
import os

import os, urllib.request, urllib.parse

def load_dataset(filename, module_name="11 - Polars"):
    """
    Carga o descarga de forma segura el dataset para ejecución local o en Google Colab.
    Si no se encuentra localmente ni en GitHub, lo genera automáticamente.
    """
    candidates = [
        os.path.join("data", filename),
        os.path.join(module_name, "data", filename),
        os.path.join("..", "data", filename),
        os.path.join("..", module_name, "data", filename),
        os.path.join("Data Science programming", module_name, "data", filename),
        os.path.join("..", "Data Science programming", module_name, "data", filename),
        filename
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
            
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    folder_path = f"Data Science programming/{module_name}"
    encoded_folder = urllib.parse.quote(folder_path)
    encoded_file = urllib.parse.quote(filename)
    
    urls = [
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{encoded_folder}/data/{encoded_file}",
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/master/{encoded_folder}/data/{encoded_file}"
    ]
    
    print(f"📥 Intentando descargar '{filename}' desde el repositorio oficial...")
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                if response.status == 200:
                    with open(target_path, 'wb') as out_f:
                        out_f.write(response.read())
                    print(f"✅ Dataset '{filename}' descargado exitosamente.")
                    return target_path
        except Exception:
            continue
            
    print(f"⚙️ Generando '{filename}' sintéticamente para ejecución inmediata...")
    import polars as pl
    import numpy as np
    np.random.seed(42)
    
    n_clientes = 1000
    df_c = pl.DataFrame({
        'id_cliente': [f'CLI-{i:04d}' for i in range(1, n_clientes + 1)],
        'nombre': [f'Cliente_{i}' for i in range(1, n_clientes + 1)],
        'segmento': np.random.choice(['Corporativo', 'Pyme', 'Consumo', 'Gobierno'], n_clientes),
        'edad': np.random.randint(18, 70, n_clientes),
        'ciudad_residencia': np.random.choice(['Tunja', 'Bogotá', 'Medellín', 'Cali', 'Bucaramanga'], n_clientes),
        'ingreso_anual': np.random.normal(45000000, 15000000, n_clientes).round(2)
    })
    
    n_ventas = 60000
    cats = ['Tecnología', 'Mobiliario', 'Material de Oficina', 'Servicios']
    prods = ['Laptop Pro', 'Monitor 4K', 'Silla Ergonómica', 'Escritorio', 'Papel A4', 'Tóner', 'Mantenimiento']
    ciudades = ['Tunja', 'Bogotá', 'Medellín', 'Cali', 'Barranquilla']
    
    cant = np.random.randint(1, 10, n_ventas)
    pu = np.random.choice([25000.0, 120000.0, 450000.0, 1200000.0, 3500000.0], n_ventas)
    desc = np.random.choice([0.0, 0.05, 0.10, 0.15], n_ventas)
    tot = (cant * pu * (1 - desc)).round(2)
    
    df_v = pl.DataFrame({
        'id_venta': [f'VNT-{i:06d}' for i in range(1, n_ventas + 1)],
        'fecha': [f'2024-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}' for _ in range(n_ventas)],
        'id_cliente': np.random.choice(df_c['id_cliente'], n_ventas),
        'categoria': np.random.choice(cats, n_ventas),
        'producto': np.random.choice(prods, n_ventas),
        'cantidad': cant,
        'precio_unitario': pu,
        'descuento': desc,
        'ciudad_venta': np.random.choice(ciudades, n_ventas),
        'total_venta': tot
    })
    
    c_csv_path = os.path.join("data", "clientes.csv")
    c_pq_path = os.path.join("data", "clientes.parquet")
    v_csv_path = os.path.join("data", "ventas.csv")
    v_pq_path = os.path.join("data", "ventas.parquet")
    
    if not os.path.exists(c_csv_path): df_c.write_csv(c_csv_path)
    if not os.path.exists(c_pq_path): df_c.write_parquet(c_pq_path)
    if not os.path.exists(v_csv_path): df_v.write_csv(v_csv_path)
    if not os.path.exists(v_pq_path): df_v.write_parquet(v_pq_path)
    
    print(f"✅ Datasets preparados exitosamente en '{target_path}'.")
    return target_path

ruta_csv = load_dataset("ventas.csv")
df = pl.read_csv(ruta_csv)
print(f"🚀 Polars versión: {pl.__version__}")
print(f"Datos cargados: {df.shape}")

🚀 Polars versión: 1.35.2
Datos cargados: (60000, 10)


---
## 2. Contexto 1: `select()` — Proyección Paralela 🎯

Cada expresión dentro de `select` se ejecuta de forma concurrente en distintos hilos de la CPU:

In [48]:
res_select = df.select([
    pl.col("producto"),
    pl.col("cantidad"),
    pl.col("precio_unitario"),
    (pl.col("cantidad") * pl.col("precio_unitario")).alias("subtotal_bruto"),
    (pl.col("descuento") * 100).round(1).alias("porcentaje_descuento")
])
display(res_select.head(5))

producto,cantidad,precio_unitario,subtotal_bruto,porcentaje_descuento
str,i64,f64,f64,f64
"""Laptop Pro""",7,25000.0,175000.0,5.0
"""Monitor 4K""",9,120000.0,1.08e6,0.0
"""Laptop Pro""",5,25000.0,125000.0,15.0
"""Laptop Pro""",4,450000.0,1.8e6,5.0
"""Laptop Pro""",8,25000.0,200000.0,0.0


---
## 3. Selección Avanzada: Múltiples Columnas, Patrones y Exclusión 🧩

`pl.col()` no solo acepta un nombre: también acepta **varios nombres a la vez**, un **patrón de texto** (entre `^` y `$`, como una expresión regular) e incluso puedes pedir "todas menos estas" con `pl.exclude()`. Esto evita escribir listas larguísimas de columnas a mano:

| Forma | Qué selecciona |
|---|---|
| `pl.col("a", "b", "c")` | Exactamente esas columnas, por nombre. |
| `pl.col("^.*_venta$")` | Cualquier columna cuyo nombre haga match con el patrón (aquí, que termine en `_venta`). |
| `pl.all()` | Todas las columnas del DataFrame. |
| `pl.exclude("a", "b")` | Todas las columnas **excepto** las indicadas. |

In [49]:
# Todas las columnas cuyo nombre termina en "_venta" (id_venta, ciudad_venta, total_venta)
res_regex = df.select(pl.col("^.*_venta$"))
print("Columnas seleccionadas por patron:", res_regex.columns)
display(res_regex.head(3))

# Todo el DataFrame, menos las columnas de texto largo
res_sin_texto = df.select(pl.exclude("producto", "ciudad_venta"))
print("\nColumnas tras excluir 'producto' y 'ciudad_venta':", res_sin_texto.columns)

Columnas seleccionadas por patron: ['id_venta', 'ciudad_venta', 'total_venta']


id_venta,ciudad_venta,total_venta
str,str,f64
"""VNT-000001""","""Bogotá""",166250.0
"""VNT-000002""","""Barranquilla""",1.08e6
"""VNT-000003""","""Barranquilla""",106250.0



Columnas tras excluir 'producto' y 'ciudad_venta': ['id_venta', 'fecha', 'id_cliente', 'categoria', 'cantidad', 'precio_unitario', 'descuento', 'total_venta']


---
## 4. Contexto 2: `with_columns()` — Agregar Nuevas Columnas ➕

A diferencia de `select()`, `with_columns()` **conserva todas las columnas originales** y solo añade (o reemplaza, si usas el mismo nombre) las que definas. Aquí también aparece `pl.when().then().otherwise()`, el equivalente de Polars a un `CASE WHEN` de SQL o a una cadena de `if/elif/else` — pero vectorizado y evaluado en paralelo para las 60,000 filas a la vez:

In [50]:
df_enriquecido = df.with_columns([
    (pl.col("precio_unitario") * (1 - pl.col("descuento"))).alias("precio_con_descuento"),
    pl.when(pl.col("total_venta") >= 5000000)
      .then(pl.lit("Alta Venta"))
      .when(pl.col("total_venta") >= 1000000)
      .then(pl.lit("Media Venta"))
      .otherwise(pl.lit("Baja Venta"))
      .alias("segmento_ticket")
])
display(df_enriquecido.select(["id_venta", "producto", "total_venta", "segmento_ticket"]).head(5))

id_venta,producto,total_venta,segmento_ticket
str,str,f64,str
"""VNT-000001""","""Laptop Pro""",166250.0,"""Baja Venta"""
"""VNT-000002""","""Monitor 4K""",1.08e6,"""Media Venta"""
"""VNT-000003""","""Laptop Pro""",106250.0,"""Baja Venta"""
"""VNT-000004""","""Laptop Pro""",1.71e6,"""Media Venta"""
"""VNT-000005""","""Laptop Pro""",200000.0,"""Baja Venta"""


---
## 5. Más `with_columns()`: Combinando Texto y Booleanos con `pl.format()` 📝

`with_columns()` acepta tantas expresiones nuevas como necesites en una sola llamada, y cada una puede depender de varias columnas a la vez. `pl.format()` funciona como un f-string vectorizado: arma un texto por fila combinando columnas y literales.

In [51]:
df_resumen_texto = df.with_columns([
    pl.format("{}-{} ({} u.)", pl.col("categoria"), pl.col("producto"), pl.col("cantidad")).alias("descripcion_corta"),
    (pl.col("descuento") > 0).alias("tiene_descuento")
])
display(df_resumen_texto.select(["descripcion_corta", "tiene_descuento"]).head(5))

descripcion_corta,tiene_descuento
str,bool
"""Mobiliario-Laptop Pro (7 u.)""",true
"""Mobiliario-Monitor 4K (9 u.)""",false
"""Mobiliario-Laptop Pro (5 u.)""",true
"""Mobiliario-Laptop Pro (4 u.)""",true
"""Servicios-Laptop Pro (8 u.)""",false


---
## 6. Contexto 3: `filter()` — Filtrado Booleano Vectorizado 🔎

`filter()` recibe una **expresión booleana** (una máscara: `True`/`False` por fila) y conserva únicamente las filas donde el resultado es `True`. Es el equivalente al `WHERE` de SQL.

In [52]:
# Filtrar ventas de Tecnologia en Tunja con cantidad mayor a 5
filtro = df.filter(
    (pl.col("categoria") == "Tecnología") &
    (pl.col("ciudad_venta") == "Tunja") &
    (pl.col("cantidad") > 5)
)
print(f"Registros filtrados: {filtro.height}")
display(filtro.head(4))

Registros filtrados: 1359


id_venta,fecha,id_cliente,categoria,producto,cantidad,precio_unitario,descuento,ciudad_venta,total_venta
str,str,str,str,str,i64,f64,f64,str,f64
"""VNT-000017""","""2024-03-16""","""CLI-0346""","""Tecnología""","""Laptop Pro""",6,25000.0,0.05,"""Tunja""",142500.0
"""VNT-000020""","""2024-02-02""","""CLI-0981""","""Tecnología""","""Silla Ergonómica""",7,25000.0,0.0,"""Tunja""",175000.0
"""VNT-000033""","""2024-01-01""","""CLI-0872""","""Tecnología""","""Tóner""",7,3.5e6,0.05,"""Tunja""",2.3275e7
"""VNT-000058""","""2024-09-23""","""CLI-0813""","""Tecnología""","""Laptop Pro""",8,3.5e6,0.05,"""Tunja""",2.66e7


---
## 7. Combinando Condiciones: `&`, `|` e `.is_in()` 🧮

Los operadores lógicos de Polars son `&` (AND), `|` (OR) y `~` (NOT) — **no** `and`/`or`/`not` de Python, que no funcionan sobre expresiones vectorizadas. Cada condición compuesta debe ir entre paréntesis. `.is_in([...])` reemplaza cadenas largas de `== "a" | == "b" | == "c"`:

In [53]:
# Ventas grandes (>15 unidades) en categorias tecnologicas, O cualquier venta hecha en Bogota
filtro_or = df.filter(
    ((pl.col("cantidad") > 15) & pl.col("categoria").is_in(["Tecnología", "Hardware"]))
    | (pl.col("ciudad_venta") == "Bogotá")
)
print(f"Registros que cumplen la condición combinada: {filtro_or.height}")
display(filtro_or.head(4))

Registros que cumplen la condición combinada: 12043


id_venta,fecha,id_cliente,categoria,producto,cantidad,precio_unitario,descuento,ciudad_venta,total_venta
str,str,str,str,str,i64,f64,f64,str,f64
"""VNT-000001""","""2024-11-09""","""CLI-0117""","""Mobiliario""","""Laptop Pro""",7,25000.0,0.05,"""Bogotá""",166250.0
"""VNT-000006""","""2024-02-23""","""CLI-0644""","""Material de Oficina""","""Tóner""",9,450000.0,0.15,"""Bogotá""",3.4425e6
"""VNT-000011""","""2024-11-01""","""CLI-0023""","""Tecnología""","""Papel A4""",1,25000.0,0.15,"""Bogotá""",21250.0
"""VNT-000012""","""2024-06-01""","""CLI-0535""","""Servicios""","""Escritorio""",7,450000.0,0.0,"""Bogotá""",3.15e6


---
## 8. Los 3 Contextos, Uno al Lado del Otro 🔀

Aunque los tres reciben expresiones con la misma sintaxis (`pl.col(...)`), cada uno responde una pregunta distinta:

| Contexto | ¿Qué le das? | ¿Qué te devuelve? | ¿Cambia el número de filas? |
|---|---|---|---|
| **`select(...)`** | Las columnas/cálculos que quieres **ver** | Solo esas columnas | No |
| **`with_columns(...)`** | Columnas nuevas o que reemplazan existentes | Todo lo que ya había + lo nuevo | No |
| **`filter(...)`** | Una condición booleana | Todas las columnas, pero solo las filas que cumplen | Sí (puede reducir filas) |

Piénsalo como tres **modos** de una misma herramienta: modo "muéstrame esto" (`select`), modo "agrégale esto sin tocar lo demás" (`with_columns`) y modo "quédate solo con esto" (`filter`). Cambias de modo según la pregunta que quieras responder, pero el vocabulario de expresiones (`pl.col()`, operadores, `when/then/otherwise`) es el mismo en los tres.

---
## 9. Cadenas de Texto: El Namespace `.str.*` 🔤

Todas las operaciones de texto viven bajo el "espacio de nombres" `.str` de una expresión de columna tipo `String`. Esto mantiene el autocompletado limpio: `pl.col("producto").str.` te muestra solo métodos de texto.

In [54]:
df_strings = df.select([
    pl.col("producto"),
    pl.col("producto").str.to_uppercase().alias("producto_mayus"),
    pl.col("producto").str.len_chars().alias("longitud_nombre"),
    pl.col("producto").str.contains("(?i)mesa").alias("es_mesa"),
    pl.col("producto").str.starts_with("Monitor").alias("es_monitor"),
    pl.col("categoria").str.replace("Tecnología", "Tech").alias("categoria_corta"),
    pl.col("categoria").str.slice(0, 3).alias("categoria_abrev")
])
display(df_strings.head(5))

producto,producto_mayus,longitud_nombre,es_mesa,es_monitor,categoria_corta,categoria_abrev
str,str,u32,bool,bool,str,str
"""Laptop Pro""","""LAPTOP PRO""",10,false,false,"""Mobiliario""","""Mob"""
"""Monitor 4K""","""MONITOR 4K""",10,false,true,"""Mobiliario""","""Mob"""
"""Laptop Pro""","""LAPTOP PRO""",10,false,false,"""Mobiliario""","""Mob"""
"""Laptop Pro""","""LAPTOP PRO""",10,false,false,"""Mobiliario""","""Mob"""
"""Laptop Pro""","""LAPTOP PRO""",10,false,false,"""Servicios""","""Ser"""


---
## 10. Fechas y Horas: El Namespace `.dt.*` 📅

`fecha` llega desde el CSV como texto plano (`String`), porque un CSV no guarda metadatos de tipo. El primer paso siempre es convertirla con `.str.to_datetime()`; solo después queda disponible el namespace `.dt` con año, mes, día de la semana, trimestre, etc.

In [55]:
df_tiempo = df.with_columns([
    pl.col("fecha").str.to_datetime()
]).with_columns([
    pl.col("fecha").dt.year().alias("anio"),
    pl.col("fecha").dt.month().alias("mes"),
    pl.col("fecha").dt.weekday().alias("dia_semana"),
    pl.col("fecha").dt.quarter().alias("trimestre"),
    pl.col("fecha").dt.date().alias("solo_fecha")
])
display(df_tiempo.select(["fecha", "anio", "mes", "trimestre", "dia_semana", "solo_fecha"]).head(4))

# Una vez que 'anio' existe como columna, se puede filtrar por año normalmente
ventas_2024 = df_tiempo.filter(pl.col("anio") == 2024)
print(f"\nVentas registradas en 2024: {ventas_2024.height}")

fecha,anio,mes,trimestre,dia_semana,solo_fecha
datetime[μs],i32,i8,i8,i8,date
2024-11-09 00:00:00,2024,11,4,6,2024-11-09
2024-04-10 00:00:00,2024,4,2,3,2024-04-10
2024-05-27 00:00:00,2024,5,2,1,2024-05-27
2024-10-15 00:00:00,2024,10,4,2,2024-10-15



Ventas registradas en 2024: 60000


---
## 11. Uniendo Todo: una Mini-Pipeline con `filter → with_columns → select` 🔗

En la práctica, casi nunca usas un solo contexto de forma aislada: los encadenas. Como cada paso devuelve un `DataFrame` nuevo, puedes leerlos como una receta de arriba hacia abajo:

In [56]:
pipeline = (
    df
    .with_columns(pl.col("fecha").str.to_datetime())
    .filter(pl.col("fecha").dt.year() == 2024)
    .with_columns([
        (pl.col("total_venta") * (1 - pl.col("descuento"))).alias("total_neto"),
        pl.col("producto").str.to_uppercase().alias("producto_mayus")
    ])
    .select(["id_venta", "fecha", "producto_mayus", "categoria", "total_neto"])
    .sort("total_neto", descending=True)
)
display(pipeline.head(5))

id_venta,fecha,producto_mayus,categoria,total_neto
str,datetime[μs],str,str,f64
"""VNT-000060""",2024-12-27 00:00:00,"""SILLA ERGONÓMICA""","""Servicios""",3.15e7
"""VNT-000168""",2024-07-28 00:00:00,"""TÓNER""","""Servicios""",3.15e7
"""VNT-000210""",2024-08-16 00:00:00,"""MANTENIMIENTO""","""Material de Oficina""",3.15e7
"""VNT-000412""",2024-02-11 00:00:00,"""ESCRITORIO""","""Mobiliario""",3.15e7
"""VNT-000611""",2024-12-04 00:00:00,"""LAPTOP PRO""","""Tecnología""",3.15e7


---
## 12. Ejercicio Práctico: Ventas de Servicios Cloud con IVA 🧪

Usando el DataFrame `df` (ventas) que cargamos al inicio, resuelve lo siguiente:

1. **`filter()`**: quédate solo con las ventas de la categoría `"Servicios Cloud"` que tengan `cantidad >= 3`.
2. **`with_columns()`**: agrega una columna `total_con_iva` calculada como `total_venta * 1.19`, redondeada a 2 decimales con `.round(2)`.
3. **`select()` + `sort()`**: selecciona `id_venta`, `producto` y `total_con_iva`, y ordénalas de mayor a menor `total_con_iva`. Muestra las primeras 5 filas.

Escribe tu solución en la celda de abajo antes de revisar la respuesta guiada:

In [57]:
# 1. Filtrar Servicios Cloud con cantidad >= 3
# paso1 = ...

# 2. Agregar columna total_con_iva
# paso2 = ...

# 3. Seleccionar, ordenar y mostrar el top 5
# resultado = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
resultado = (
    df
    .filter((pl.col("categoria") == "Servicios Cloud") & (pl.col("cantidad") >= 3))
    .with_columns((pl.col("total_venta") * 1.19).round(2).alias("total_con_iva"))
    .select(["id_venta", "producto", "total_con_iva"])
    .sort("total_con_iva", descending=True)
)
display(resultado.head(5))
```
</details>

---
## 13. Resumen y Próximos Pasos 📌

| Concepto | Idea Clave |
|---|---|
| **Expresión** | Una "receta" declarativa (`pl.col("x") * 2`) que no hace nada hasta que se coloca en un contexto. |
| **`select()`** | Proyecta columnas/cálculos nuevos; descarta el resto. |
| **`with_columns()`** | Conserva todo y añade/reemplaza columnas. |
| **`filter()`** | Conserva todas las columnas, pero solo las filas que cumplen una condición booleana. |
| **`pl.col()` avanzado** | Acepta varios nombres, patrones tipo regex, y se combina con `pl.all()` / `pl.exclude()`. |
| **`when/then/otherwise`** | El `CASE WHEN` vectorizado de Polars para lógica condicional. |
| **`&`, `\|`, `~`, `.is_in()`** | Operadores para combinar condiciones booleanas (nunca `and`/`or`/`not` de Python). |
| **`.str.*`** | Namespace para operaciones de texto: mayúsculas, longitud, `contains`, `replace`, `slice`... |
| **`.dt.*`** | Namespace para fechas: año, mes, día de la semana, trimestre... (requiere `.str.to_datetime()` primero si la columna llegó como texto). |

➡️ **Siguiente paso:** en el **Cuaderno 02 — Agrupaciones, Joins y Funciones de Ventana** aprenderás a resumir estos datos por grupos (`group_by`), combinarlos con la tabla de `clientes` (`join`) y calcular métricas "sin perder la fila" con `.over()`.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Módulo Extra: Polars de Alto Rendimiento</i>
  </p>
</div>